# 05 - Strong Ablation Official Mini

Required: component, model, K inference, LR, and H=8 vs H=16 if both prepared roots are available.

This version also adds a lightweight checkpoint-stage analysis: pretrain-only vs posttrain-task-specific, if Notebook 03 produced both checkpoints.


In [ ]:
!pip install -q pandas pyarrow numpy tqdm matplotlib

In [ ]:
from pathlib import Path
import json, math, time, random
import numpy as np, pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Beta

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
SMOKE_TEST=False; BATCH_SIZE=256; EVAL_BATCH_SIZE=512; EPOCHS=1; NUM_WORKERS=2; USE_AMP=True
MAX_TRAIN_SAMPLES=2048 if SMOKE_TEST else None; MAX_EVAL_SAMPLES=1024 if SMOKE_TEST else None; MAX_STEPS_PER_EPOCH=20 if SMOKE_TEST else None
OUTPUT_ROOT=Path("/kaggle/working/gr00t_official_ablation"); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUNS=[
 {"run":"state_vl_layers8_lr1e4","type":"component/model/hyperparam","use_vl":True,"layers":8,"hidden":512,"lr":1e-4,"H":16},
 {"run":"state_only_layers8_lr1e4","type":"component","use_vl":False,"layers":8,"hidden":512,"lr":1e-4,"H":16},
 {"run":"state_vl_layers4_lr1e4","type":"model","use_vl":True,"layers":4,"hidden":512,"lr":1e-4,"H":16},
 {"run":"state_vl_layers8_lr5e5","type":"hyperparam","use_vl":True,"layers":8,"hidden":512,"lr":5e-5,"H":16},
 {"run":"state_vl_H8_lr1e4","type":"horizon","use_vl":True,"layers":8,"hidden":512,"lr":1e-4,"H":8},
]
K_VALUES=[1,4,8,16]

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, hidden_dim, buckets=1000):
        super().__init__()
        self.buckets = buckets
        self.embed = nn.Embedding(buckets, hidden_dim)
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
    def forward(self, t):
        idx = torch.clamp((t * self.buckets).long(), 0, self.buckets - 1)
        return self.mlp(self.embed(idx))

class CrossBlock(nn.Module):
    def __init__(self, hidden_dim, heads, dropout):
        super().__init__()
        self.n1 = nn.LayerNorm(hidden_dim)
        self.sa = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.n2 = nn.LayerNorm(hidden_dim)
        self.ca = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.n3 = nn.LayerNorm(hidden_dim)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout)
        )
    def forward(self, x, mem):
        y = self.n1(x)
        x = x + self.sa(y, y, y, need_weights=False)[0]
        y = self.n2(x)
        x = x + self.ca(y, mem, mem, need_weights=False)[0]
        return x + self.ff(self.n3(x))

class OfficialMiniVLDiT(nn.Module):
    def __init__(self, state_dim, action_dim, state_horizon, action_horizon, vl_dim,
                 hidden_dim=512, layers=8, heads=8, dropout=0.1, buckets=1000):
        super().__init__()
        self.state_horizon = state_horizon
        self.action_horizon = action_horizon
        self.state_proj = nn.Linear(state_dim, hidden_dim)
        self.action_proj = nn.Linear(action_dim, hidden_dim)
        self.vl_proj = nn.Linear(vl_dim, hidden_dim)
        self.time = TimeEmbedding(hidden_dim, buckets)
        self.state_pos = nn.Embedding(state_horizon, hidden_dim)
        self.action_pos = nn.Embedding(action_horizon, hidden_dim)
        self.blocks = nn.ModuleList([CrossBlock(hidden_dim, heads, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(hidden_dim)
        self.out = nn.Linear(hidden_dim, action_dim)
    def forward(self, noisy_action, state_history, vl_features, t):
        s_pos = self.state_pos(torch.arange(self.state_horizon, device=noisy_action.device))[None]
        a_pos = self.action_pos(torch.arange(self.action_horizon, device=noisy_action.device))[None]
        s = self.state_proj(state_history) + s_pos
        a = self.action_proj(noisy_action) + a_pos
        x = torch.cat([s, a], dim=1) + self.time(t)[:, None, :]
        mem = self.vl_proj(vl_features)
        for block in self.blocks:
            x = block(x, mem)
        return self.out(self.norm(x[:, self.state_horizon:]))

In [ ]:
def find_dir(name, required):
    cand=[]
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if base.exists(): cand.extend(base.rglob(name))
    for c in sorted(cand, key=str):
        if c.is_dir() and all((c/r).exists() for r in required): return c
    return None
def root(H): return find_dir(f"gr00t_prepared_official_H{H}", ["states_train.npy","actions_train_chunk.npy","action_mask_train.npy","normalization_stats.json"])
def feature_root(H): return find_dir(f"official_mini_vldit_H{H}", ["config.json"])
print("roots:", {h: root(h) for h in [8,16]})

In [ ]:
class DS(Dataset):
    def __init__(self, prepared, fr, split, H, use_vl, max_samples):
        self.H=H; self.use_vl=use_vl; self.prepared=prepared
        self.states=np.load(prepared/f"states_{split}.npy", mmap_mode="r"); self.actions=np.load(prepared/f"actions_{split}_chunk.npy", mmap_mode="r"); self.masks=np.load(prepared/f"action_mask_{split}.npy", mmap_mode="r")
        self.stats=json.loads((prepared/"normalization_stats.json").read_text()); self.sm=np.asarray(self.stats["state_mean"],np.float32); self.ss=np.asarray(self.stats["state_std"],np.float32); self.am=np.asarray(self.stats["action_mean"],np.float32); self.asd=np.asarray(self.stats["action_std"],np.float32)
        if use_vl:
            if fr is None or not (fr.parent/f"vl_features_{split}.npy").exists(): raise FileNotFoundError(f"Missing VL features for H={H}. Run Notebook 03 for this H.")
            self.features=np.load(fr.parent/f"vl_features_{split}.npy",mmap_mode="r"); self.index=pd.read_parquet(fr.parent/f"vl_feature_index_{split}.parquet").sort_values("feature_index").reset_index(drop=True); self.vl_dim=int(self.features.shape[-1])
        else:
            samples=pd.read_parquet(prepared/f"{split}_samples.parquet"); self.index=pd.DataFrame({"sample_id":samples.sample_id.values}); self.vl_dim=1
        if max_samples is not None and len(self.index)>max_samples: self.index=self.index.sample(n=int(max_samples),random_state=SEED).reset_index(drop=True)
    def __len__(self): return len(self.index)
    def __getitem__(self,i):
        r=self.index.iloc[i]; sid=int(r.sample_id); state=(np.asarray(self.states[sid],np.float32)-self.sm)/self.ss; action=(np.asarray(self.actions[sid],np.float32)-self.am)/self.asd; mask=np.asarray(self.masks[sid],np.float32)
        vl=np.asarray(self.features[int(r.feature_index)],np.float32) if self.use_vl else np.zeros((1,1),np.float32)
        return {"state":torch.from_numpy(state),"action":torch.from_numpy(action),"mask":torch.from_numpy(mask),"vl":torch.from_numpy(vl)}
def collate(batch): return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

In [ ]:
def train_eval(run):
    H=run["H"]; pr=root(H); fr=feature_root(H)
    if pr is None: return {"run":run["run"],"status":"skipped","reason":f"missing prepared H{H}"}
    try:
        tr=DS(pr,fr,"train",H,run["use_vl"],MAX_TRAIN_SAMPLES); ev=DS(pr,fr,"test",H,run["use_vl"],MAX_EVAL_SAMPLES)
    except Exception as exc:
        return {"run":run["run"],"status":"skipped","reason":str(exc)}
    tl=DataLoader(tr,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),drop_last=True,collate_fn=collate); el=DataLoader(ev,batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),collate_fn=collate)
    # Moi ablation run khoi tao model moi tu dau de so sanh cong bang.
    model=OfficialMiniVLDiT(44,44,1,H,tr.vl_dim,run["hidden"],run["layers"],8,.1).to(device); opt=torch.optim.AdamW(model.parameters(),lr=run["lr"],weight_decay=1e-5,betas=(.95,.999)); scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available()); beta=Beta(torch.tensor(1.5,device=device),torch.tensor(1.0,device=device))
    losses=[]; start=time.time(); steps=0; model.train(); max_steps=len(tl) if MAX_STEPS_PER_EPOCH is None else min(MAX_STEPS_PER_EPOCH,len(tl))
    for i,b in enumerate(tqdm(tl,total=max_steps,desc=run["run"])):
        if i>=max_steps: break
        state=b["state"].to(device); action=b["action"].to(device); mask=b["mask"].to(device); vl=b["vl"].to(device); t=beta.sample((action.size(0),)).to(device=device,dtype=action.dtype); noise=torch.randn_like(action); noisy=(1-t[:,None,None])*noise+t[:,None,None]*action; target=action-noise
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):
            pred=model(noisy,state,vl,t); loss=((pred-target).pow(2)*mask).sum()/(mask.sum()+1e-6)
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu())); steps+=1
    def sample(state,vl,K):
        x=torch.randn(state.size(0),H,44,device=state.device,dtype=state.dtype); dt=1.0/K
        for k in range(K): x=x+dt*model(x,state,vl,torch.full((state.size(0),),k/K,device=state.device,dtype=state.dtype))
        return x
    am=torch.tensor(ev.am,device=device)[None,None,:]; ast=torch.tensor(ev.asd,device=device)[None,None,:]; k_rows=[]; model.eval()
    with torch.no_grad():
        # K-step Euler inference: doi K nhung giu nguyen checkpoint de do trade-off toc do/chat luong.
        for K in K_VALUES:
            sq=ab=count=samples=0
            for b in tqdm(el,desc=f"eval {run['run']} K={K}"):
                state=b["state"].to(device); action=b["action"].to(device); mask=b["mask"].to(device); vl=b["vl"].to(device)
                pred=sample(state,vl,K); diff=(pred.float()*ast+am)-(action.float()*ast+am)
                sq+=float((diff.square()*mask).sum().cpu()); ab+=float((diff.abs()*mask).sum().cpu()); count+=int(mask.sum().cpu()); samples+=int(action.size(0))
            mse=sq/max(count,1); k_rows.append({"K":K,"raw_mse":mse,"raw_mae":ab/max(count,1),"raw_rmse":math.sqrt(mse),"eval_samples":samples})
    best=next(x for x in k_rows if x["K"]==4)
    return {"run":run["run"],"status":"completed","type":run["type"],"use_vl":run["use_vl"],"H":H,"layers":run["layers"],"hidden_dim":run["hidden"],"lr":run["lr"],"global_steps":steps,"train_last_loss":losses[-1] if losses else None,"train_elapsed_sec":round(time.time()-start,2),**best,"k_results":k_rows}

results=[]
for run in RUNS:
    res=train_eval(run); print(res); results.append(res)
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
def eval_existing_checkpoint_stage(label, ckpt_name):
    H = 16; pr = root(H); fr = feature_root(H)
    if pr is None or fr is None:
        return {"run": label, "status": "skipped", "type": "checkpoint_stage", "reason": "missing prepared root or feature root"}
    ckpt_path = fr / ckpt_name
    if not ckpt_path.exists():
        return {"run": label, "status": "skipped", "type": "checkpoint_stage", "reason": f"missing {ckpt_name}"}
    try:
        ev = DS(pr, fr, "test", H, True, MAX_EVAL_SAMPLES)
        el = DataLoader(ev, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=torch.cuda.is_available(), collate_fn=collate)
        ckpt = torch.load(ckpt_path, map_location=device); cfg = ckpt["config"]
        model = OfficialMiniVLDiT(44,44,cfg["state_horizon"],cfg["action_horizon"],cfg["vl_feature_dim"],cfg["hidden_dim"],cfg["num_layers"],cfg["num_heads"],cfg["dropout"]).to(device)
        model.load_state_dict(ckpt["model_state_dict"]); model.eval()
        am=torch.tensor(ev.am,device=device)[None,None,:]; ast=torch.tensor(ev.asd,device=device)[None,None,:]
        def sample_stage(state,vl,K=4):
            x=torch.randn(state.size(0),H,44,device=state.device,dtype=state.dtype); dt=1.0/K
            for k in range(K): x=x+dt*model(x,state,vl,torch.full((state.size(0),),k/K,device=state.device,dtype=state.dtype))
            return x
        sq=ab=count=samples=0
        with torch.no_grad():
            for b in tqdm(el,desc=f"eval {label}"):
                state=b["state"].to(device); action=b["action"].to(device); mask=b["mask"].to(device); vl=b["vl"].to(device)
                pred=sample_stage(state,vl,4); diff=(pred.float()*ast+am)-(action.float()*ast+am)
                sq+=float((diff.square()*mask).sum().cpu()); ab+=float((diff.abs()*mask).sum().cpu()); count+=int(mask.sum().cpu()); samples+=int(action.size(0))
        mse=sq/max(count,1)
        return {"run": label, "status": "completed", "type": "checkpoint_stage", "use_vl": True, "H": H,
                "layers": cfg.get("num_layers"), "hidden_dim": cfg.get("hidden_dim"), "lr": cfg.get("learning_rate"),
                "global_steps": int(ckpt.get("global_step", 0)), "raw_mse": mse, "raw_mae": ab/max(count,1),
                "raw_rmse": math.sqrt(mse), "eval_samples": samples, "K": 4, "checkpoint": ckpt_name}
    except Exception as exc:
        return {"run": label, "status": "skipped", "type": "checkpoint_stage", "reason": repr(exc)}
    finally:
        if torch.cuda.is_available(): torch.cuda.empty_cache()

stage_results = [
    eval_existing_checkpoint_stage("pretrain_only", "pretrain_checkpoint.pt"),
    eval_existing_checkpoint_stage("posttrain_task_specific", "posttrain_checkpoint.pt"),
]
stage_df = pd.DataFrame(stage_results)
stage_df.to_csv(OUTPUT_ROOT/"pretrain_posttrain_comparison.csv", index=False)

summary_df=pd.DataFrame([{k:v for k,v in r.items() if k!="k_results"} for r in results])
summary_df = pd.concat([summary_df, stage_df], ignore_index=True, sort=False)
summary_df.to_csv(OUTPUT_ROOT/"ablation_summary.csv",index=False)
k_rows=[]
for r in results:
    if r.get("status")=="completed":
        for kr in r["k_results"]: k_rows.append({"run":r["run"],**kr})
pd.DataFrame(k_rows).to_csv(OUTPUT_ROOT/"k_step_tradeoff.csv",index=False)
display(summary_df); display(stage_df)
completed=summary_df[summary_df.status=="completed"] if "status" in summary_df else pd.DataFrame()
plot_df=completed[completed.type != "checkpoint_stage"] if len(completed) and "type" in completed else completed
if len(plot_df):
    plt.figure(figsize=(9,4)); plt.bar(plot_df.run, plot_df.raw_mse); plt.xticks(rotation=30, ha="right"); plt.ylabel("Raw MSE @K=4"); plt.tight_layout(); plt.savefig(OUTPUT_ROOT/"ablation_bar_chart.png",dpi=160); plt.show()
summary={"status":"completed","num_runs":len(results),"num_completed":int(len(completed)),
         "required_ablation_types":["component","model","inference_K","hyperparam","horizon_H8_vs_H16"],
         "additional_analysis":["pretrain_only_vs_posttrain_task_specific"],
         "metrics_space":"denormalized_raw_action_space","uses_action_mask":True,
         "results":results, "pretrain_posttrain_comparison":stage_results}
(OUTPUT_ROOT/"ablation_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps({"num_completed":len(completed),"output_root":str(OUTPUT_ROOT),"stage_analysis":stage_results}, indent=2))
